# Fingerprint: An End-to-End ML Framework for LLM Fingerprinting

---

## Notebook 01 — Feature Engineering

---

### Purpose
Transform preprocessed text corpora into numerical feature matrices suitable
for machine learning classifiers. This notebook produces four distinct feature
representations that will be independently evaluated in subsequent modelling notebooks.

### Objectives
1. Extract **TF-IDF** features (word and character n-gram level)
2. Extract **Character N-Gram** features (multiple granularities, combined)
3. Extract **Stylometric** features (lexical, punctuation, structural, syntactic, readability)
4. Extract **Sentence Embedding** features (dense 384-dim vectors via MiniLM)
5. Compare feature spaces statistically
6. Save all feature matrices for downstream modelling

### Workflow
```
Preprocessed Parquet Files
        │
        ▼
  Dataset Loading & Verification
        │
        ├── TF-IDF Extraction ──────────────────► data/features/tfidf/
        ├── Character N-Gram Extraction ────────► data/features/char/
        ├── Stylometric Extraction ─────────────► data/features/style/
        └── Sentence Embedding Extraction ──────► data/features/embedding/
```

### Expected Outputs
| Feature Set | Files |
|---|---|
| TF-IDF (word) | `data/features/tfidf/tfidf_fingerprint.npz` |
| TF-IDF (char) | `data/features/tfidf/char_tfidf_fingerprint.npz` |
| Char N-Gram | `data/features/char/char_fingerprint.npz` |
| Stylometric | `data/features/style/style_fingerprint.npz` |
| Embeddings | `data/features/embedding/emb_fingerprint.npz` |

### Dependencies
```
scikit-learn >= 1.2.2
sentence-transformers
numpy, pandas, scipy
matplotlib, plotly
```

### Notebook Outline
1. Project Overview
2. Import Libraries
3. Configuration
4. Path Definitions
5. Dataset Loading
6. Dataset Verification
7. Feature Engineering Overview
8. TF-IDF Feature Engineering
9. Character N-Gram Feature Engineering
10. Stylometric Feature Engineering
11. Sentence Embedding Feature Engineering
12. Feature Statistics
13. Feature Comparison
14. Save Feature Matrices
15. Notebook Summary

---

## 1. Project Overview

### Project: Fingerprint

**Goal**: Identify which Large Language Model (LLM) generated a piece of text using Machine Learning.

**Completed Stages**:
- ✅ Dataset Engineering
- ✅ Synthetic Data Generation
- ✅ Dataset Management
- ✅ Exploratory Data Analysis
- ✅ NLP Preprocessing

**Current Stage**: Feature Engineering

---

### Why Feature Engineering Matters for LLM Fingerprinting

Different LLMs have distinct **stylistic fingerprints**:
- **Vocabulary patterns** → TF-IDF captures word and character frequency distributions
- **Sub-word patterns** → Character n-grams capture spelling conventions and tokenisation artefacts
- **Writing style** → Stylometric features capture sentence structure, punctuation habits, readability
- **Semantic fingerprint** → Sentence embeddings capture meaning-level signature patterns

By generating all four representations, we can empirically compare which feature
space best discriminates between LLM sources.

---

## 2. Import Libraries

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import sys
import logging
import warnings
from pathlib import Path

# Suppress verbose deprecation warnings during development
warnings.filterwarnings('ignore')

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# ── Project root resolution ───────────────────────────────────────────────────
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Project source modules ────────────────────────────────────────────────────
from src.feature_engineering.utils import (
    FeatureEngineeringConfig,
    setup_logger,
    save_feature_matrix,
    encode_labels,
)
from src.feature_engineering.tfidf_extractor import TFIDFExtractor
from src.feature_engineering.char_ngram_extractor import CharNGramExtractor
from src.feature_engineering.stylometric_extractor import StylometricExtractor
from src.feature_engineering.embedding_extractor import EmbeddingExtractor
from src.utils.helpers import set_global_seed, make_output_dirs

print('✅ All libraries imported successfully.')

---

## 3. Configuration

In [ ]:
# ── Load configuration from YAML ──────────────────────────────────────────────
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'feature_engineering.yaml'

config = FeatureEngineeringConfig.from_yaml(CONFIG_PATH)

# ── Setup logging ─────────────────────────────────────────────────────────────
setup_logger(
    log_file=str(PROJECT_ROOT / config.log_file),
    level=config.log_level,
)
logger = logging.getLogger(__name__)

# ── Set reproducibility seed ──────────────────────────────────────────────────
set_global_seed(config.random_seed)

# ── Display configuration summary ─────────────────────────────────────────────
print(f'Project      : {config.project_name}')
print(f'Stage        : {config.stage}')
print(f'Random Seed  : {config.random_seed}')
print(f'Text Column  : {config.text_col}')
print(f'Label Column : {config.label_col}')

---

## 4. Path Definitions

In [ ]:
# ── Input paths (preprocessed parquet files) ──────────────────────────────────
PATH_PIPELINE_A = PROJECT_ROOT / config.pipeline_a_path   # Fingerprint-Preserving
PATH_PIPELINE_B = PROJECT_ROOT / config.pipeline_b_path   # Traditional NLP

# ── Output feature directories ────────────────────────────────────────────────
DIR_TFIDF = PROJECT_ROOT / config.tfidf_dir
DIR_CHAR  = PROJECT_ROOT / config.char_dir
DIR_STYLE = PROJECT_ROOT / config.style_dir
DIR_EMB   = PROJECT_ROOT / config.emb_dir

# ── Create output directories ─────────────────────────────────────────────────
make_output_dirs(DIR_TFIDF, DIR_CHAR, DIR_STYLE, DIR_EMB)

print('Input Paths:')
print(f'  Pipeline A (Fingerprint): {PATH_PIPELINE_A}')
print(f'  Pipeline B (Traditional): {PATH_PIPELINE_B}')
print()
print('Output Directories:')
for name, path in [('TF-IDF', DIR_TFIDF), ('Char', DIR_CHAR),
                   ('Style', DIR_STYLE), ('Embedding', DIR_EMB)]:
    print(f'  {name:12s}: {path}')

---

## 5. Dataset Loading

In [ ]:
# ── Load Pipeline A — Fingerprint-Preserving ──────────────────────────────────
df_a = pd.read_parquet(PATH_PIPELINE_A)
print(f'Pipeline A loaded: {df_a.shape[0]:,} rows × {df_a.shape[1]} columns')
df_a.head(3)

In [ ]:
# ── Load Pipeline B — Traditional NLP ────────────────────────────────────────
df_b = pd.read_parquet(PATH_PIPELINE_B)
print(f'Pipeline B loaded: {df_b.shape[0]:,} rows × {df_b.shape[1]} columns')
df_b.head(3)

---

## 6. Dataset Verification

In [ ]:
# ── Verify required columns exist ─────────────────────────────────────────────
TEXT_COL  = config.text_col
LABEL_COL = config.label_col

for pipeline_name, df in [('Pipeline A', df_a), ('Pipeline B', df_b)]:
    missing = [c for c in [TEXT_COL, LABEL_COL] if c not in df.columns]
    if missing:
        raise ValueError(f'{pipeline_name}: Missing columns {missing}')
    null_texts  = df[TEXT_COL].isnull().sum()
    null_labels = df[LABEL_COL].isnull().sum()
    n_classes   = df[LABEL_COL].nunique()
    print(f'{pipeline_name}:')
    print(f'  Rows         : {len(df):,}')
    print(f'  Null texts   : {null_texts}')
    print(f'  Null labels  : {null_labels}')
    print(f'  Classes      : {n_classes}')
    print(f'  Class list   : {sorted(df[LABEL_COL].unique())}')
    print()

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
label_counts = df_a[LABEL_COL].value_counts().reset_index()
label_counts.columns = ['Model', 'Count']

fig = px.bar(
    label_counts,
    x='Model',
    y='Count',
    title='Class Distribution — Pipeline A (Fingerprint-Preserving)',
    color='Model',
    template='plotly_dark',
)
fig.show()

# ── Encode labels (Pipeline A is authoritative for class list) ─────────────
y_a, le = encode_labels(df_a[LABEL_COL])
CLASS_NAMES = le.classes_
print(f'Classes ({len(CLASS_NAMES)}): {list(CLASS_NAMES)}')

---

## 7. Feature Engineering Overview

### Feature Representation Strategy

| # | Feature Set | Type | Dimensionality | Captures |
|---|---|---|---|---|
| 1 | **TF-IDF Word** | Sparse | ~50,000 | Word frequency & importance |
| 2 | **TF-IDF Char** | Sparse | ~30,000 | Character-level n-gram frequencies |
| 3 | **Char N-Grams** | Sparse | ~30,000+ | Sub-word patterns, tokenisation artifacts |
| 4 | **Stylometric** | Dense | ~30 | Writing style, punctuation, readability |
| 5 | **Embeddings** | Dense | 384 | Semantic fingerprint (sentence-level) |

### Dual Pipeline Strategy

All features are extracted from **both** preprocessed pipelines:
- **Pipeline A** (Fingerprint-Preserving): Minimal cleaning — preserves stylistic signals
- **Pipeline B** (Traditional NLP): Aggressive normalisation — lemmatised, stopwords removed

This dual extraction allows a rigorous scientific comparison of which preprocessing
strategy yields better classification performance.

---

## 8. TF-IDF Feature Engineering

### 8.1 Why TF-IDF for LLM Fingerprinting?

Term Frequency-Inverse Document Frequency is a classical NLP weighting scheme that
assigns higher importance to terms that are distinctive to specific documents or classes
while down-weighting ubiquitous terms. For LLM fingerprinting:
- Each LLM has characteristic vocabulary preferences
- Character TF-IDF captures sub-word spelling conventions (e.g. British vs American spellings)
- The `sublinear_tf=True` setting reduces the dominance of high-frequency terms

In [ ]:
# ── Instantiate TF-IDF extractor ──────────────────────────────────────────────
tfidf_extractor = TFIDFExtractor(cfg=config.tfidf_cfg)

print('TF-IDF Configuration:')
print(f"  Word max_features : {config.tfidf_cfg['word']['max_features']:,}")
print(f"  Word ngram_range  : {config.tfidf_cfg['word']['ngram_range']}")
print(f"  Char max_features : {config.tfidf_cfg['char']['max_features']:,}")
print(f"  Char ngram_range  : {config.tfidf_cfg['char']['ngram_range']}")

In [ ]:
# ── Extract TF-IDF features — Pipeline A (Fingerprint-Preserving) ─────────────
texts_a = df_a[TEXT_COL].fillna('')

word_matrix_a, char_tfidf_matrix_a = tfidf_extractor.fit_transform(texts_a)

# Save vectorizer for reuse
tfidf_extractor.save(DIR_TFIDF)

print(f'Word TF-IDF Matrix A : {word_matrix_a.shape}')
print(f'Char TF-IDF Matrix A : {char_tfidf_matrix_a.shape}')

In [ ]:
# ── Extract TF-IDF features — Pipeline B (Traditional NLP) ────────────────────
texts_b = df_b[TEXT_COL].fillna('')
y_b, _ = encode_labels(df_b[LABEL_COL])

# Note: Transform using the SAME vectorizer fitted on Pipeline A
# This ensures feature spaces are aligned for fair comparison
word_matrix_b, char_tfidf_matrix_b = tfidf_extractor.transform(texts_b)

print(f'Word TF-IDF Matrix B : {word_matrix_b.shape}')
print(f'Char TF-IDF Matrix B : {char_tfidf_matrix_b.shape}')

In [ ]:
# ── TF-IDF statistics summary ─────────────────────────────────────────────────
tfidf_stats = tfidf_extractor.get_statistics(word_matrix_a, char_tfidf_matrix_a)

print('TF-IDF Feature Statistics:')
for key, val in tfidf_stats.items():
    print(f'  {key:25s}: {val}')

---

## 9. Character N-Gram Feature Engineering

### 9.1 Why Character N-Grams?

Character-level n-grams are widely considered the most discriminative single
feature type for authorship attribution. For LLM fingerprinting they capture:
- **Tokenisation artifacts**: Different models were trained on different tokenisers
- **Morphological preferences**: Particular word endings, prefixes
- **Punctuation micro-patterns**: How the model spaces around punctuation
- **Noise robustness**: Less sensitive to vocabulary drift than word-level features

Multiple n-gram ranges are extracted and stacked for maximum coverage.

In [ ]:
# ── Instantiate Char N-Gram extractor ─────────────────────────────────────────
char_extractor = CharNGramExtractor(cfg=config.char_ngrams_cfg)

print('Char N-Gram Configuration:')
print(f"  N-gram ranges  : {config.char_ngrams_cfg['ngram_ranges']}")
print(f"  Max features   : {config.char_ngrams_cfg['max_features']:,}")
print(f"  Combine ranges : {config.char_ngrams_cfg['combine']}")

In [ ]:
# ── Extract char n-gram features — Pipeline A ─────────────────────────────────
char_matrix_a = char_extractor.fit_transform(texts_a)
char_extractor.save(DIR_CHAR)

char_stats_a = char_extractor.get_statistics(char_matrix_a)
print('Char N-Gram Matrix A Statistics:')
for key, val in char_stats_a.items():
    print(f'  {key:25s}: {val}')

In [ ]:
# ── Extract char n-gram features — Pipeline B ─────────────────────────────────
char_matrix_b = char_extractor.transform(texts_b)
print(f'Char N-Gram Matrix B : {char_matrix_b.shape}')

---

## 10. Stylometric Feature Engineering

### 10.1 Stylometric Features — Motivation

Stylometric features capture **human-interpretable writing style signals**:

| Feature Group | Examples | LLM Signal |
|---|---|---|
| **Lexical** | TTR, hapax ratio, avg word length | Vocabulary diversity |
| **Punctuation** | Comma/period ratios | Punctuation habits |
| **Structural** | Paragraph count, sentence count | Document organisation |
| **Syntactic** | Question density, pronoun ratio | Rhetorical style |
| **Readability** | Flesch-Kincaid grade | Complexity preferences |

These are **entirely interpretable** — every feature has a clear linguistic meaning.
This makes them valuable for both classification and for understanding *what* makes
each LLM's writing style distinctive.

In [ ]:
# ── Instantiate Stylometric extractor ─────────────────────────────────────────
style_extractor = StylometricExtractor(cfg=config.stylometric_cfg)

# ── Extract stylometric features — Pipeline A (use fingerprint-preserving text) 
style_matrix_a = style_extractor.fit_transform(texts_a)

print(f'Stylometric Matrix A : {style_matrix_a.shape}')
print(f'Feature names ({len(style_extractor.feature_names_)}): {style_extractor.feature_names_[:8]} ...')

In [ ]:
# ── Extract stylometric features — Pipeline B ─────────────────────────────────
# NOTE: Stylometric features should be extracted from Pipeline A text only
# because Pipeline B strips punctuation and casing, destroying stylometric signals.
# We use Pipeline A text for both and simply align labels.
style_matrix_b = style_extractor.fit_transform(
    df_a[TEXT_COL].fillna('')   # Intentional: preserve style signals
)
print(f'Stylometric Matrix B : {style_matrix_b.shape}')

In [ ]:
# ── Visualise stylometric feature distributions by LLM ────────────────────────
style_df = style_extractor.get_feature_dataframe(texts_a)
style_df[LABEL_COL] = df_a[LABEL_COL].values

# Plot average word length by model
fig = px.box(
    style_df,
    x=LABEL_COL,
    y='avg_word_length',
    color=LABEL_COL,
    title='Average Word Length Distribution by LLM Source',
    template='plotly_dark',
)
fig.show()

In [ ]:
# ── Stylometric statistics ─────────────────────────────────────────────────────
style_stats = style_extractor.get_statistics(style_matrix_a)
print(f"Stylometric shape : {style_stats['shape']}")
print(f"N features        : {style_stats['n_features']}")
print()
print('Feature means (first 10):')
means = style_stats['feature_means']
for fname, mean_val in list(means.items())[:10]:
    print(f'  {fname:30s}: {mean_val:.4f}')

---

## 11. Sentence Embedding Feature Engineering

### 11.1 Sentence Embeddings — Motivation

The `all-MiniLM-L6-v2` sentence transformer encodes each document into a
384-dimensional dense vector that captures **semantic meaning**.

For LLM fingerprinting, sentence embeddings capture:
- **Topic and reasoning patterns** unique to each model's training
- **Response structure patterns** — how different models frame their answers
- **Semantic granularity** — some models produce more abstract, others more concrete text

> ⚠️ **Note**: This step is computationally intensive. On CPU, expect ~5-15 minutes
> for a large corpus. Switch `device: "cuda"` in `configs/feature_engineering.yaml`
> if GPU is available.

In [ ]:
# ── Instantiate Embedding extractor ───────────────────────────────────────────
emb_extractor = EmbeddingExtractor(cfg=config.embeddings_cfg)

print(f'Model      : {emb_extractor.model_name}')
print(f'Batch size : {emb_extractor.batch_size}')
print(f'Max length : {emb_extractor.max_seq_length} tokens')
print(f'Normalize  : {emb_extractor.normalize}')
print(f'Device     : {emb_extractor.device}')

In [ ]:
# ── Encode Pipeline A texts ────────────────────────────────────────────────────
emb_matrix_a = emb_extractor.fit_transform(texts_a)

emb_stats_a = emb_extractor.get_statistics(emb_matrix_a)
print('Embedding Matrix A Statistics:')
for key, val in emb_stats_a.items():
    print(f'  {key:20s}: {val}')

In [ ]:
# ── Encode Pipeline B texts ────────────────────────────────────────────────────
emb_matrix_b = emb_extractor.transform(texts_b)
print(f'Embedding Matrix B : {emb_matrix_b.shape}')

---

## 12. Feature Statistics

In [ ]:
# ── Consolidated feature space summary table ───────────────────────────────────
import scipy.sparse as sp

def sparsity(m):
    """Compute sparsity ratio of a matrix."""
    if sp.issparse(m):
        return 1 - m.nnz / (m.shape[0] * m.shape[1])
    return 0.0  # Dense matrices have zero sparsity

summary = pd.DataFrame([
    {'Feature Set': 'TF-IDF Word',   'Shape': word_matrix_a.shape,
     'Type': 'Sparse', 'Sparsity': f'{sparsity(word_matrix_a):.4%}',
     'Pipeline': 'A (Fingerprint)'},
    {'Feature Set': 'TF-IDF Char',   'Shape': char_tfidf_matrix_a.shape,
     'Type': 'Sparse', 'Sparsity': f'{sparsity(char_tfidf_matrix_a):.4%}',
     'Pipeline': 'A (Fingerprint)'},
    {'Feature Set': 'Char N-Grams',  'Shape': char_matrix_a.shape,
     'Type': 'Sparse', 'Sparsity': f'{sparsity(char_matrix_a):.4%}',
     'Pipeline': 'A (Fingerprint)'},
    {'Feature Set': 'Stylometric',   'Shape': style_matrix_a.shape,
     'Type': 'Dense',  'Sparsity': 'N/A',
     'Pipeline': 'A (Fingerprint)'},
    {'Feature Set': 'Embeddings',    'Shape': emb_matrix_a.shape,
     'Type': 'Dense',  'Sparsity': 'N/A',
     'Pipeline': 'A (Fingerprint)'},
])

summary

---

## 13. Feature Comparison

In [ ]:
# ── Dimensionality comparison bar chart ───────────────────────────────────────
feat_dims = pd.DataFrame({
    'Feature Set': ['TF-IDF Word', 'TF-IDF Char', 'Char N-Grams',
                    'Stylometric', 'Embeddings'],
    'Dimensions':  [
        word_matrix_a.shape[1],
        char_tfidf_matrix_a.shape[1],
        char_matrix_a.shape[1],
        style_matrix_a.shape[1],
        emb_matrix_a.shape[1],
    ]
})

fig = px.bar(
    feat_dims,
    x='Feature Set',
    y='Dimensions',
    title='Feature Space Dimensionality Comparison',
    color='Feature Set',
    template='plotly_dark',
    log_y=True,
)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# ── Embedding PCA visualisation (2D) ──────────────────────────────────────────
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=config.random_seed)
emb_2d = pca.fit_transform(emb_matrix_a)

pca_df = pd.DataFrame({
    'PC1': emb_2d[:, 0],
    'PC2': emb_2d[:, 1],
    'Model': df_a[LABEL_COL].values
})

fig = px.scatter(
    pca_df,
    x='PC1',
    y='PC2',
    color='Model',
    title='Sentence Embeddings — PCA 2D Projection by LLM Source',
    template='plotly_dark',
    opacity=0.6,
)
fig.show()

---

## 14. Save Feature Matrices

In [ ]:
# ── Save TF-IDF Word features ──────────────────────────────────────────────────
save_feature_matrix(word_matrix_a, y_a, le, DIR_TFIDF, 'tfidf_fingerprint')
save_feature_matrix(word_matrix_b, y_b, le, DIR_TFIDF, 'tfidf_traditional')
print('✅ TF-IDF Word features saved.')

In [ ]:
# ── Save TF-IDF Char features ──────────────────────────────────────────────────
save_feature_matrix(char_tfidf_matrix_a, y_a, le, DIR_TFIDF, 'char_tfidf_fingerprint')
save_feature_matrix(char_tfidf_matrix_b, y_b, le, DIR_TFIDF, 'char_tfidf_traditional')
print('✅ TF-IDF Char features saved.')

In [ ]:
# ── Save Char N-Gram features ──────────────────────────────────────────────────
save_feature_matrix(char_matrix_a, y_a, le, DIR_CHAR, 'char_fingerprint')
save_feature_matrix(char_matrix_b, y_b, le, DIR_CHAR, 'char_traditional')
print('✅ Char N-Gram features saved.')

In [ ]:
# ── Save Stylometric features ──────────────────────────────────────────────────
save_feature_matrix(style_matrix_a, y_a, le, DIR_STYLE, 'style_fingerprint')
save_feature_matrix(style_matrix_b, y_b, le, DIR_STYLE, 'style_traditional')
print('✅ Stylometric features saved.')

In [ ]:
# ── Save Embedding features ────────────────────────────────────────────────────
emb_extractor.save_matrix(emb_matrix_a, DIR_EMB / 'emb_fingerprint.npz')
emb_extractor.save_matrix(emb_matrix_b, DIR_EMB / 'emb_traditional.npz')
np.save(DIR_EMB / 'labels_emb_fingerprint.npy', y_a)
np.save(DIR_EMB / 'labels_emb_traditional.npy', y_b)
np.save(DIR_EMB / 'classes_emb_fingerprint.npy', CLASS_NAMES)
print('✅ Embedding features saved.')

In [ ]:
# ── Final verification — list all saved files ──────────────────────────────────
from pathlib import Path

for dir_path, dir_name in [
    (DIR_TFIDF, 'TF-IDF'), (DIR_CHAR, 'Char N-Gram'),
    (DIR_STYLE, 'Stylometric'), (DIR_EMB, 'Embedding')
]:
    files = sorted(Path(dir_path).iterdir())
    print(f'\n{dir_name} ({dir_path}):')
    for f in files:
        size_mb = f.stat().st_size / 1024 / 1024
        print(f'  {f.name:<50s}  {size_mb:.2f} MB')

---

## 15. Notebook Summary

### What Was Accomplished

| Step | Status | Output |
|---|---|---|
| TF-IDF Word Extraction | ✅ | `data/features/tfidf/tfidf_*.npz` |
| TF-IDF Char Extraction | ✅ | `data/features/tfidf/char_tfidf_*.npz` |
| Char N-Gram Extraction | ✅ | `data/features/char/char_*.npz` |
| Stylometric Extraction | ✅ | `data/features/style/style_*.npz` |
| Sentence Embedding Extraction | ✅ | `data/features/embedding/emb_*.npz` |
| Label Encoding | ✅ | `*_labels.npy`, `*_classes.npy` |
| Vectorizer Persistence | ✅ | `*.joblib` |

### Next Steps

→ **Notebook 02**: Model Selection — rationale for choosing Logistic Regression,
  Linear SVM, Random Forest, and XGBoost

→ **Notebooks 03–06**: Individual model training and evaluation on each feature set

---

*Fingerprint Project — Feature Engineering Stage — Complete*